# Dicionário empírico otimizado das tabelas DB2

## Objetivo

Estudar **isoladamente** a estrutura, preenchimento, cardinalidade e valores observados nas tabelas, construindo um dicionário empírico de códigos a partir dos próprios dados.

Não há cliente específico, Água, transferências, consentimento, conta própria, regras financeiras ou joins entre tabelas.

## Fontes principais

1. `DB2GFP.TRAN_RLZD_INST_PCT`
2. `DB2GFP.INF_OPB_CT_CLI`
3. `DB2GFP.CMPT_TRAN_RLZD_CC`
4. `DB2GFP.CTGR_TRAN_OPB`
5. `DB2GFP.GR_CTGR_TRAN`

## Estudo adicional isolado

- `DB2OPB.AUTZ_ATV_CPTO_CLI`

A tabela adicional não possui documentação específica encontrada nos arquivos do Projeto. Portanto, ela é estudada apenas empiricamente, sem inferência de significado a partir do nome.

## Política de volume

- `TRAN_RLZD_INST_PCT`: **3 últimos meses completos calculados automaticamente a partir de `HOJE`**. Com `HOJE=2026-08-10`, o recorte é `DT_TRAN >= 2026-05-01` e `< 2026-08-01` (maio, junho e julho completos).
- Demais tabelas: política adaptativa.
  - primeiro tenta obter `COUNT(*)` com timeout controlado;
  - se o volume for tecnicamente aceitável, usa tabela completa;
  - se o volume exceder o limite ou a contagem não puder ser obtida, usa amostra técnica limitada e registra isso no Markdown.

## Otimização de performance

A versão anterior disparava um shuffle de cardinalidade para praticamente cada coluna.

Esta versão usa:

1. **uma carga de conteúdo por tabela**;
2. `persist(MEMORY_AND_DISK)` apenas enquanto a tabela é reutilizada;
3. **uma agregação em lote** para nulos, datas, estatísticas básicas e cardinalidade aproximada de todas as colunas;
4. cardinalidade exata apenas para colunas cuja triagem indica possível domínio (`<=100`);
5. frequências somente para colunas de baixa/média cardinalidade;
6. associações código ↔ texto somente entre poucos pares candidatos da própria tabela;
7. `unpersist()` ao concluir cada fonte.

## Saída

Único arquivo:

`estudo_tabelas_resultado.md`

## Escopo mantido

A otimização **não reduz o conteúdo do estudo**: permanecem classificação automática das colunas, nulos/preenchimento, cardinalidade, frequências, estatísticas técnicas, associações código ↔ texto intratabela, dicionário pendente e dicionário inferido.

In [ ]:
# ============================================================
# 1. Bootstrap local — padrão do Projeto
# ============================================================
from pathlib import Path

OUTPUT_MD = Path("estudo_tabelas_resultado.md")
spark = None
gerenciador_spark = None

def gravar_falha_bootstrap(etapa, exc):
    OUTPUT_MD.write_text(
        "# Estudo das tabelas\n\n"
        "## Observações técnicas\n\n"
        f"- **EXECUÇÃO INCOMPLETA** — {etapa}: {type(exc).__name__}.\n",
        encoding="utf-8",
    )

try:
    from src.utils.gerenciador_sessao_spark_local import (
        GerenciadorSessaoSpark,
        ler_variavel_ambiente_local,
    )

    ambiente = ler_variavel_ambiente_local("AMBIENTE").upper()
    if ambiente != "MODELAGEM":
        ambiente = "PRODUCAO"

    gerenciador_spark = GerenciadorSessaoSpark(
        nome_sessao="estudo_dicionario_empirico_otimizado",
        adicionar_variaveis={
            "DOMINIO": "t2i",
            "AMBIENTE": ambiente,
        },
        nome_arquivo_env_modelagem="desenv.env",
        exibir_configuracao=False,
        ativar_logs=True,
    )

    spark = gerenciador_spark.criar_sessao_spark(
        db2=True,
        driver_memory="12g",
        driver_cores=4,
        executor_memory="12g",
        executor_cores=4,
        num_executors=8,
        spark_conf={
            "spark.driver.memoryOverhead": "8g",
            "spark.executor.memoryOverhead": "4g",
            "spark.serializer": "org.apache.spark.serializer.KryoSerializer",
            "spark.kryoserializer.buffer.max": "512m",
            "spark.sql.adaptive.enabled": "true",
            "spark.sql.adaptive.coalescePartitions.enabled": "true",
            "spark.sql.adaptive.skewJoin.enabled": "true",
            "spark.sql.adaptive.localShuffleReader.enabled": "true",
            "spark.sql.shuffle.partitions": "240",
            "spark.sql.sources.partitionOverwriteMode": "dynamic",
            "spark.hadoop.mapreduce.input.fileinputformat.input.dir.recursive": "true",
            "spark.sql.broadcastTimeout": "8000",
            "spark.executor.heartbeatInterval": "30s",
            "spark.network.timeout": "300s",
            "spark.sql.session.timeZone": "America/Sao_Paulo",
        },
    )

    print("[OK] Sessão Spark criada.")
except Exception as exc:
    gravar_falha_bootstrap("bootstrap local", exc)
    raise

In [ ]:
# ============================================================
# 2. Abstrações remotas já existentes no Projeto
# ============================================================
try:
    if spark is None:
        raise RuntimeError("Sessão Spark indisponível.")

    %run ./src/utils/gerenciador_sessao_spark_remoto.ipynb
except Exception as exc:
    gravar_falha_bootstrap(
        "carga do gerenciador Spark remoto",
        exc,
    )
    raise

In [ ]:
%%spark

# ============================================================
# 3. Cliente DB2 e configuração do estudo
# ============================================================
import os
import re
from collections import defaultdict
from datetime import datetime, date

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StringType,
    DateType,
    TimestampType,
    BooleanType,
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
)
from pyspark.storagelevel import StorageLevel


cliente_db2 = criar_cliente_db2_spark(env=dict(os.environ))

# ------------------------------------------------------------
# Janela temporal: 3 últimos meses COMPLETOS
# HOJE já é enviado automaticamente pelo GerenciadorSessaoSpark.
# Ex.: HOJE=2026-08-10 -> 2026-05-01 até < 2026-08-01.
# ------------------------------------------------------------
HOJE_ESTUDO = datetime.strptime(
    ler_variavel_ambiente_spark("HOJE"),
    "%Y-%m-%d",
).date()

PRIMEIRO_DIA_MES_ATUAL = date(
    HOJE_ESTUDO.year,
    HOJE_ESTUDO.month,
    1,
)

def deslocar_primeiro_dia_mes(data_ref, meses):
    indice = data_ref.year * 12 + (data_ref.month - 1) + int(meses)
    return date(
        indice // 12,
        indice % 12 + 1,
        1,
    )

DT_TRAN_INICIO = deslocar_primeiro_dia_mes(
    PRIMEIRO_DIA_MES_ATUAL,
    -3,
)
DT_TRAN_FIM_EXCLUSIVO = PRIMEIRO_DIA_MES_ATUAL

PERIODO_TRAN = (
    f"{DT_TRAN_INICIO.isoformat()} até "
    f"< {DT_TRAN_FIM_EXCLUSIVO.isoformat()}"
)


FONTES_PRINCIPAIS = [
    "DB2GFP.TRAN_RLZD_INST_PCT",
    "DB2GFP.INF_OPB_CT_CLI",
    "DB2GFP.CMPT_TRAN_RLZD_CC",
    "DB2GFP.CTGR_TRAN_OPB",
    "DB2GFP.GR_CTGR_TRAN",
]

FONTE_ADICIONAL = "DB2OPB.AUTZ_ATV_CPTO_CLI"

TABELAS = FONTES_PRINCIPAIS + [FONTE_ADICIONAL]

# Limites técnicos configuráveis.
LIMITE_TABELA_COMPLETA = 5_000_000
LIMITE_AMOSTRA_TECNICA = 1_000_000
TIMEOUT_CONTAGEM_VOLUME = 180

# Triagem de cardinalidade.
APPROX_RSD = 0.02
LIMIAR_EXATO_CANDIDATO = 200
LOTE_EXATO = 12

CONFIG = {
    "DB2GFP.TRAN_RLZD_INST_PCT": {
        "grupo": "PRINCIPAL",
        "modo": "RECORTE_FIXO",
        "universo_planejado": f"3 últimos meses completos por DT_TRAN: {PERIODO_TRAN}",
        "where": (
            f"DT_TRAN >= DATE('{DT_TRAN_INICIO.isoformat()}') "
            f"AND DT_TRAN < DATE('{DT_TRAN_FIM_EXCLUSIVO.isoformat()}')"
        ),
        "documentacao_projeto": "DISPONÍVEL",
    },
    "DB2GFP.INF_OPB_CT_CLI": {
        "grupo": "PRINCIPAL",
        "modo": "ADAPTATIVO",
        "universo_planejado": "tabela completa se tecnicamente aceitável",
        "where": None,
        "documentacao_projeto": "DISPONÍVEL",
    },
    "DB2GFP.CMPT_TRAN_RLZD_CC": {
        "grupo": "PRINCIPAL",
        "modo": "ADAPTATIVO",
        "universo_planejado": "tabela completa se tecnicamente aceitável",
        "where": None,
        "documentacao_projeto": "DISPONÍVEL",
    },
    "DB2GFP.CTGR_TRAN_OPB": {
        "grupo": "PRINCIPAL",
        "modo": "ADAPTATIVO",
        "universo_planejado": "tabela completa se tecnicamente aceitável",
        "where": None,
        "documentacao_projeto": "DISPONÍVEL",
    },
    "DB2GFP.GR_CTGR_TRAN": {
        "grupo": "PRINCIPAL",
        "modo": "ADAPTATIVO",
        "universo_planejado": "tabela completa se tecnicamente aceitável",
        "where": None,
        "documentacao_projeto": "DISPONÍVEL",
    },
    "DB2OPB.AUTZ_ATV_CPTO_CLI": {
        "grupo": "ADICIONAL",
        "modo": "ADAPTATIVO",
        "universo_planejado": (
            "estudo empírico isolado; tabela completa se tecnicamente aceitável"
        ),
        "where": None,
        "documentacao_projeto": "NÃO ENCONTRADA",
    },
}

RESULTADO_ESTUDO = {
    "execucao_ok": True,
    "tabelas": {},
    "observacoes_tecnicas": [],
    "dicionario_pendente": [],
    "dicionario_inferido": [],
    "periodo_transacoes": {
        "hoje_referencia": HOJE_ESTUDO.isoformat(),
        "inicio": DT_TRAN_INICIO.isoformat(),
        "fim_exclusivo": DT_TRAN_FIM_EXCLUSIVO.isoformat(),
        "descricao": PERIODO_TRAN,
    },
    "parametros_performance": {
        "limite_tabela_completa": LIMITE_TABELA_COMPLETA,
        "limite_amostra_tecnica": LIMITE_AMOSTRA_TECNICA,
        "approx_rsd": APPROX_RSD,
        "limiar_exato_candidato": LIMIAR_EXATO_CANDIDATO,
    },
}

print("[OK] ClientDb2Spark disponível.")

In [ ]:
%%spark

# ============================================================
# 4. Leitura adaptativa e utilidades
# ============================================================

DATE_TYPES = (DateType, TimestampType)
NUMERIC_TYPES = (
    ByteType,
    ShortType,
    IntegerType,
    LongType,
    FloatType,
    DoubleType,
    DecimalType,
)
INTEGER_TYPES = (ByteType, ShortType, IntegerType, LongType)

TEXT_PREFIXES = (
    "TX_", "NM_", "DS_", "DCR_", "DESC_", "TX_DCR_",
)

DOMAIN_PREFIXES = (
    "CD_", "IN_", "TP_", "TIP_",
)

DOMAIN_TOKENS = (
    "EST", "STS", "FLG", "IND", "NTZ", "MOD", "MOE", "SIT",
)

IDENTIFIER_PATTERNS = [
    r"^CD_CLI$",
    r"^CD_CLI_TITR_CT$",
    r"^NR_TRAN",
    r"^NR_PTC$",
    r"IDFR",
    r"CPF",
    r"CNPJ",
    r"DOCUMENT",
    r"(^|_)DOC($|_)",
    r"PIX",
    r"AGEN",
    r"AGENCIA",
    r"(^|_)NR_CT($|_)",
    r"CONTA",
]

SENSITIVE_PATTERNS = IDENTIFIER_PATTERNS + [
    r"USUAR",
    r"LOGIN",
    r"SENHA",
    r"PASSWORD",
    r"SECRET",
    r"TOKEN",
]

def casa_padrao(nome, padroes):
    return any(
        re.search(p, str(nome), re.I)
        for p in padroes
    )

def nome_textual(nome):
    return str(nome).upper().startswith(TEXT_PREFIXES)

def nome_dominio(nome):
    n = str(nome).upper()

    if n.startswith(DOMAIN_PREFIXES):
        return True

    return any(
        re.search(rf"(^|_){token}($|_)", n)
        for token in DOMAIN_TOKENS
    )

def nome_identificador(nome):
    n = str(nome).upper()

    if casa_padrao(n, IDENTIFIER_PATTERNS):
        return True

    if n.startswith("NR_"):
        return True

    return False

def coluna_sensivel(nome):
    return casa_padrao(
        nome,
        SENSITIVE_PATTERNS,
    )

def valor_python(v):
    if v is None:
        return None

    if isinstance(v, (bool, int, float)):
        return v

    if hasattr(v, "isoformat"):
        try:
            return v.isoformat()
        except Exception:
            pass

    return str(v)

def sanitizar_texto(v):
    if v is None:
        return None

    texto = str(v)

    texto = re.sub(
        r"(?i)\b[A-Z0-9._%+-]+@[A-Z0-9.-]+\.[A-Z]{2,}\b",
        "<EMAIL_OCULTO>",
        texto,
    )
    texto = re.sub(
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
        "<DOCUMENTO_OCULTO>",
        texto,
    )
    texto = re.sub(
        r"\b\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}\b",
        "<DOCUMENTO_OCULTO>",
        texto,
    )
    texto = re.sub(
        r"(?i)\b[0-9a-f]{8}-[0-9a-f]{4}-[1-5][0-9a-f]{3}-[89ab][0-9a-f]{3}-[0-9a-f]{12}\b",
        "<CHAVE_OCULTA>",
        texto,
    )

    return texto[:1000]

def rows_dict(df, limite, texto_cols=None):
    texto_cols = set(texto_cols or [])
    saida = []

    for row in df.limit(int(limite)).collect():
        item = {}

        for k, v in row.asDict(recursive=True).items():
            if k in texto_cols:
                item[k] = sanitizar_texto(v)
            else:
                item[k] = valor_python(v)

        saida.append(item)

    return saida

def probe_tabela(tabela):
    return cliente_db2.run_select(
        f"SELECT * FROM {tabela} FETCH FIRST 1 ROW ONLY",
        fetchsize=1,
        query_timeout=120,
    )

def projecao_explicitada(probe):
    return ", ".join(probe.columns)

def contar_tabela_controlado(tabela):
    """
    Uma única consulta de volume. Se ultrapassar o timeout,
    retorna None e o estudo usa amostra técnica.
    """
    try:
        qtd_df = cliente_db2.run_select(
            f"SELECT COUNT(*) AS QT FROM {tabela}",
            fetchsize=1,
            query_timeout=TIMEOUT_CONTAGEM_VOLUME,
        )

        row = qtd_df.collect()[0]
        return int(row["QT"])

    except Exception:
        return None

def carregar_tabela(tabela):
    cfg = CONFIG[tabela]
    probe = probe_tabela(tabela)
    colunas = projecao_explicitada(probe)

    if tabela == "DB2GFP.TRAN_RLZD_INST_PCT":
        if "DT_TRAN" not in probe.columns:
            raise RuntimeError(
                "DT_TRAN não foi observada; "
                "o recorte técnico dos 3 últimos meses completos não pode ser aplicado."
            )

        sql = (
            f"SELECT {colunas} "
            f"FROM {tabela} "
            f"WHERE {cfg['where']}"
        )

        # O ClientDb2Spark do Projeto suporta particionamento JDBC.
        # Tenta usar uma coluna inteira já presente no recorte para evitar
        # concentrar a leitura de 2026 em uma única partição JDBC.
        coluna_particao = None

        for candidato in [
            "NR_TRAN_INST_PCT",
            "NR_PTC",
        ]:
            field = next(
                (
                    f
                    for f in probe.schema.fields
                    if f.name == candidato
                ),
                None,
            )

            if (
                field is not None
                and isinstance(
                    field.dataType,
                    INTEGER_TYPES,
                )
            ):
                coluna_particao = candidato
                break

        if coluna_particao:
            try:
                bounds_df = cliente_db2.run_select(
                    (
                        f"SELECT "
                        f"MIN({coluna_particao}) AS MIN_ID, "
                        f"MAX({coluna_particao}) AS MAX_ID "
                        f"FROM {tabela} "
                        f"WHERE {cfg['where']}"
                    ),
                    fetchsize=1,
                    query_timeout=300,
                )

                bounds = bounds_df.collect()[0]
                lower = bounds["MIN_ID"]
                upper = bounds["MAX_ID"]

                if (
                    lower is not None
                    and upper is not None
                    and int(lower) < int(upper)
                ):
                    df = cliente_db2.run_select(
                        sql,
                        fetchsize=10000,
                        partition_column=coluna_particao,
                        lower_bound=int(lower),
                        upper_bound=int(upper),
                        num_partitions=32,
                        query_timeout=1200,
                    )

                    return probe, df, {
                        "universo_analisado": f"3 últimos meses completos por DT_TRAN: {PERIODO_TRAN}",
                        "volume_total_tabela": None,
                        "amostra_limitada": False,
                        "estrategia": "RECORTE_3_MESES_COMPLETOS_JDBC_PARTICIONADO",
                        "coluna_particao": coluna_particao,
                        "particoes_jdbc": 32,
                    }

            except Exception:
                # Fallback seguro: mantém o recorte de 2026 e lê via JDBC padrão.
                pass

        df = cliente_db2.run_select(
            sql,
            fetchsize=10000,
            query_timeout=1200,
        )

        return probe, df, {
            "universo_analisado": f"3 últimos meses completos por DT_TRAN: {PERIODO_TRAN}",
            "volume_total_tabela": None,
            "amostra_limitada": False,
            "estrategia": "RECORTE_3_MESES_COMPLETOS_JDBC_PADRAO",
            "coluna_particao": None,
            "particoes_jdbc": None,
        }

    volume_total = contar_tabela_controlado(
        tabela
    )

    if (
        volume_total is not None
        and volume_total <= LIMITE_TABELA_COMPLETA
    ):
        sql = (
            f"SELECT {colunas} "
            f"FROM {tabela}"
        )

        df = cliente_db2.run_select(
            sql,
            fetchsize=10000,
            query_timeout=1200,
        )

        return probe, df, {
            "universo_analisado": "tabela completa",
            "volume_total_tabela": volume_total,
            "amostra_limitada": False,
            "estrategia": "TABELA_COMPLETA",
        }

    # Se o volume é alto ou não pôde ser medido no timeout,
    # evita leitura irrestrita e usa recorte técnico explícito.
    sql = (
        f"SELECT {colunas} "
        f"FROM {tabela} "
        f"FETCH FIRST {LIMITE_AMOSTRA_TECNICA} ROWS ONLY"
    )

    df = cliente_db2.run_select(
        sql,
        fetchsize=10000,
        query_timeout=1200,
    )

    motivo = (
        f"volume total {volume_total:,} > limite técnico"
        if volume_total is not None
        else "COUNT(*) não concluído no timeout técnico"
    )

    return probe, df, {
        "universo_analisado": (
            f"amostra técnica de até "
            f"{LIMITE_AMOSTRA_TECNICA:,} linhas"
        ),
        "volume_total_tabela": volume_total,
        "amostra_limitada": True,
        "estrategia": "AMOSTRA_TECNICA",
        "motivo_amostra": motivo,
    }

print("[OK] Política adaptativa de leitura carregada.")

In [ ]:
%%spark

# ============================================================
# 5. Perfil otimizado em lote
# ============================================================

def agregacao_lote(df):
    """
    Uma única action para:
    - nulos;
    - min/max;
    - média numérica;
    - comprimento de textos;
    - strings vazias/espaços;
    - cardinalidade aproximada de TODAS as colunas.
    """
    exprs = []
    meta = {}

    for i, field in enumerate(df.schema.fields):
        c = field.name

        a_null = f"N{i}"
        a_card = f"CARD{i}"

        exprs.append(
            F.sum(
                F.when(F.col(c).isNull(), 1).otherwise(0)
            ).alias(a_null)
        )
        exprs.append(
            F.approx_count_distinct(
                F.col(c),
                rsd=APPROX_RSD,
            ).alias(a_card)
        )

        meta[a_null] = (c, "nulos")
        meta[a_card] = (
            c,
            "cardinalidade_aproximada",
        )

        if isinstance(field.dataType, DATE_TYPES):
            a_min = f"MIN{i}"
            a_max = f"MAX{i}"

            exprs.extend(
                [
                    F.min(c).alias(a_min),
                    F.max(c).alias(a_max),
                ]
            )

            meta[a_min] = (c, "minimo")
            meta[a_max] = (c, "maximo")

        elif isinstance(field.dataType, NUMERIC_TYPES):
            a_min = f"MIN{i}"
            a_max = f"MAX{i}"
            a_avg = f"AVG{i}"

            exprs.extend(
                [
                    F.min(c).alias(a_min),
                    F.max(c).alias(a_max),
                    F.avg(c).alias(a_avg),
                ]
            )

            meta[a_min] = (c, "minimo")
            meta[a_max] = (c, "maximo")
            meta[a_avg] = (c, "media")

        elif isinstance(field.dataType, StringType):
            a_lmin = f"LMIN{i}"
            a_lavg = f"LAVG{i}"
            a_lmax = f"LMAX{i}"
            a_empty = f"EMPTY{i}"
            a_spaces = f"SPACES{i}"

            exprs.extend(
                [
                    F.min(
                        F.length(F.col(c))
                    ).alias(a_lmin),
                    F.avg(
                        F.length(F.col(c))
                    ).alias(a_lavg),
                    F.max(
                        F.length(F.col(c))
                    ).alias(a_lmax),
                    F.sum(
                        F.when(
                            F.col(c) == F.lit(""),
                            1,
                        ).otherwise(0)
                    ).alias(a_empty),
                    F.sum(
                        F.when(
                            (F.col(c) != F.lit(""))
                            & (
                                F.trim(F.col(c))
                                == F.lit("")
                            ),
                            1,
                        ).otherwise(0)
                    ).alias(a_spaces),
                ]
            )

            meta[a_lmin] = (
                c,
                "comprimento_min",
            )
            meta[a_lavg] = (
                c,
                "comprimento_medio",
            )
            meta[a_lmax] = (
                c,
                "comprimento_max",
            )
            meta[a_empty] = (
                c,
                "strings_vazias",
            )
            meta[a_spaces] = (
                c,
                "somente_espacos",
            )

    valores = (
        df.agg(*exprs)
        .first()
        .asDict()
    )

    base = {
        f.name: {
            "nulos": 0,
            "cardinalidade_aproximada": 0,
            "minimo": None,
            "maximo": None,
            "media": None,
            "comprimento_min": None,
            "comprimento_medio": None,
            "comprimento_max": None,
            "strings_vazias": 0,
            "somente_espacos": 0,
        }
        for f in df.schema.fields
    }

    for alias, valor in valores.items():
        coluna, metrica = meta[alias]

        if metrica in (
            "nulos",
            "cardinalidade_aproximada",
            "strings_vazias",
            "somente_espacos",
        ):
            base[coluna][metrica] = int(
                valor or 0
            )

        elif metrica in (
            "comprimento_min",
            "comprimento_max",
        ):
            base[coluna][metrica] = (
                int(valor)
                if valor is not None
                else None
            )

        elif metrica in (
            "comprimento_medio",
            "media",
        ):
            base[coluna][metrica] = (
                round(float(valor), 6)
                if valor is not None
                else None
            )

        else:
            base[coluna][metrica] = (
                valor_python(valor)
            )

    return base

def candidatos_cardinalidade_exata(
    df,
    base,
):
    candidatos = []

    for field in df.schema.fields:
        c = field.name
        aprox = base[c][
            "cardinalidade_aproximada"
        ]

        # Margem conservadora para não perder domínios próximos do limiar.
        if (
            aprox <= LIMIAR_EXATO_CANDIDATO
            or isinstance(
                field.dataType,
                BooleanType,
            )
        ):
            candidatos.append(c)

    return candidatos

def cardinalidades_exatas_em_lotes(
    df,
    candidatos,
):
    """
    Poucos jobs: countDistinct em lotes apenas para candidatos
    pré-selecionados pela cardinalidade aproximada.
    """
    resultado = {}

    for inicio in range(
        0,
        len(candidatos),
        LOTE_EXATO,
    ):
        lote = candidatos[
            inicio : inicio + LOTE_EXATO
        ]

        exprs = [
            F.countDistinct(
                F.col(c)
            ).alias(
                f"X{i}"
            )
            for i, c in enumerate(lote)
        ]

        if not exprs:
            continue

        row = (
            df.agg(*exprs)
            .first()
            .asDict()
        )

        for i, c in enumerate(lote):
            qtd = int(
                row[f"X{i}"]
                or 0
            )

            resultado[c] = (
                ">100"
                if qtd > 100
                else qtd
            )

    return resultado

def classificar_coluna(
    field,
    cardinalidade_final,
):
    nome = field.name

    if isinstance(
        field.dataType,
        DATE_TYPES,
    ):
        return "DATA / TIMESTAMP"

    if nome_textual(nome):
        return "TEXTO / DESCRIÇÃO"

    if nome_identificador(nome):
        return "IDENTIFICADOR"

    if (
        nome_dominio(nome)
        and cardinalidade_final != ">100"
    ):
        return "DOMÍNIO / CÓDIGO"

    if (
        nome_dominio(nome)
        and cardinalidade_final == ">100"
    ):
        return "IDENTIFICADOR"

    if isinstance(
        field.dataType,
        BooleanType,
    ):
        return "DOMÍNIO / CÓDIGO"

    if isinstance(
        field.dataType,
        NUMERIC_TYPES,
    ):
        if (
            cardinalidade_final != ">100"
            and int(cardinalidade_final) <= 30
        ):
            return "DOMÍNIO / CÓDIGO"

        return "NUMÉRICO CONTÍNUO"

    if isinstance(
        field.dataType,
        StringType,
    ):
        return "TEXTO / DESCRIÇÃO"

    return "IDENTIFICADOR"

def frequencias(
    df,
    coluna,
    total,
    limite,
    sanitizar=False,
):
    freq = (
        df.groupBy(coluna)
        .agg(
            F.count("*").alias(
                "quantidade"
            )
        )
        .withColumn(
            "percentual",
            F.round(
                F.col("quantidade")
                / F.lit(int(total))
                * F.lit(100.0),
                6,
            )
            if total
            else F.lit(0.0),
        )
        .orderBy(
            F.desc("quantidade"),
            F.col(coluna).asc_nulls_last(),
        )
    )

    return rows_dict(
        freq,
        limite,
        texto_cols=[coluna]
        if sanitizar
        else [],
    )

def construir_perfil(
    df,
    total,
):
    base = agregacao_lote(df)

    candidatos = (
        candidatos_cardinalidade_exata(
            df,
            base,
        )
    )

    exatas = (
        cardinalidades_exatas_em_lotes(
            df,
            candidatos,
        )
    )

    perfil = []

    for field in df.schema.fields:
        c = field.name
        aprox = base[c][
            "cardinalidade_aproximada"
        ]

        if c in exatas:
            card = exatas[c]
        else:
            # A triagem aproximada já confirmou que a coluna está
            # confortavelmente acima do limiar operacional.
            card = ">100"

        classificacao = (
            classificar_coluna(
                field,
                card,
            )
        )

        nulos = int(
            base[c]["nulos"]
            or 0
        )
        preenchidos = (
            int(total) - nulos
        )

        valores = []

        if (
            card != ">100"
            and int(card) <= 30
        ):
            valores = frequencias(
                df,
                c,
                total,
                limite=31,
                sanitizar=(
                    classificacao
                    == "TEXTO / DESCRIÇÃO"
                ),
            )

        elif (
            card != ">100"
            and int(card) <= 100
        ):
            # Para domínio, preserva os códigos raros.
            limite = (
                101
                if classificacao
                == "DOMÍNIO / CÓDIGO"
                else 30
            )

            valores = frequencias(
                df,
                c,
                total,
                limite=limite,
                sanitizar=(
                    classificacao
                    == "TEXTO / DESCRIÇÃO"
                ),
            )

        # Alta cardinalidade não dispara groupBy adicional.
        # O objetivo é registrar >100 sem outro shuffle caro.

        perfil.append(
            {
                "nome": c,
                "tipo": (
                    field.dataType
                    .simpleString()
                    .upper()
                ),
                "linhas_analisadas": int(total),
                "preenchidos": preenchidos,
                "nulos": nulos,
                "pct_preenchido": (
                    round(
                        preenchidos
                        / int(total)
                        * 100.0,
                        6,
                    )
                    if total
                    else 0.0
                ),
                "cardinalidade": card,
                "cardinalidade_aproximada_triagem": aprox,
                "classificacao": classificacao,
                "minimo": base[c]["minimo"],
                "maximo": base[c]["maximo"],
                "media": (
                    None
                    if classificacao
                    == "IDENTIFICADOR"
                    else base[c]["media"]
                ),
                "comprimento_min": base[c][
                    "comprimento_min"
                ],
                "comprimento_medio": base[c][
                    "comprimento_medio"
                ],
                "comprimento_max": base[c][
                    "comprimento_max"
                ],
                "strings_vazias": base[c][
                    "strings_vazias"
                ],
                "somente_espacos": base[c][
                    "somente_espacos"
                ],
                "valores": valores,
            }
        )

    return perfil

print("[OK] Perfil otimizado em lote carregado.")

In [ ]:
%%spark

# ============================================================
# 6. Associação código ↔ texto intratabela
# ============================================================

CODE_PREFIXES_STEM = (
    "CD_", "TP_", "TIP_", "IN_",
)

TEXT_PREFIXES_STEM = (
    "TX_DCR_",
    "DESC_",
    "DCR_",
    "TX_",
    "NM_",
    "DS_",
)

def remover_prefixo(
    nome,
    prefixos,
):
    n = str(nome).upper()

    for prefixo in sorted(
        prefixos,
        key=len,
        reverse=True,
    ):
        if n.startswith(prefixo):
            return n[
                len(prefixo):
            ]

    return n

def tokens_nome(nome):
    return [
        t
        for t in re.split(
            r"[_\W]+",
            str(nome).upper(),
        )
        if t
    ]

def score_par(
    coluna_codigo,
    coluna_texto,
):
    a = remover_prefixo(
        coluna_codigo,
        CODE_PREFIXES_STEM,
    )
    b = remover_prefixo(
        coluna_texto,
        TEXT_PREFIXES_STEM,
    )

    if a == b:
        return 100

    if (
        a.endswith(b)
        or b.endswith(a)
    ) and min(
        len(a),
        len(b),
    ) >= 4:
        return 90

    ta = set(tokens_nome(a))
    tb = set(tokens_nome(b))

    if not ta or not tb:
        return 0

    inter = ta & tb
    uniao = ta | tb
    jaccard = (
        len(inter)
        / len(uniao)
    )

    if (
        len(inter) >= 2
        and jaccard >= 0.66
    ):
        return 80

    if (
        len(inter) >= 2
        and jaccard >= 0.50
    ):
        return 70

    return 0

def candidatos_associacao(perfil):
    idx = {
        p["nome"]: p
        for p in perfil
    }

    dominios = [
        p
        for p in perfil
        if (
            p["classificacao"]
            == "DOMÍNIO / CÓDIGO"
            and p["cardinalidade"]
            != ">100"
        )
    ]

    textos = [
        p
        for p in perfil
        if (
            p["classificacao"]
            == "TEXTO / DESCRIÇÃO"
            and nome_textual(p["nome"])
            and not coluna_sensivel(
                p["nome"]
            )
        )
    ]

    saida = []

    for dom in dominios:
        candidatos = []

        for txt in textos:
            # Se texto é claramente massivo, evita shuffle desnecessário.
            aprox_txt = txt[
                "cardinalidade_aproximada_triagem"
            ]

            teto_texto = max(
                300,
                int(
                    dom[
                        "cardinalidade"
                    ]
                )
                * 8,
            )

            if aprox_txt > teto_texto:
                continue

            score = score_par(
                dom["nome"],
                txt["nome"],
            )

            if score >= 70:
                candidatos.append(
                    (
                        score,
                        txt["nome"],
                    )
                )

        candidatos.sort(
            key=lambda x: (
                -x[0],
                x[1],
            )
        )

        # No máximo dois pares por código.
        for score, texto in candidatos[:2]:
            saida.append(
                {
                    "codigo": dom["nome"],
                    "texto": texto,
                    "score_nome": score,
                }
            )

    return saida

def chave_valor(v):
    if v is None:
        return "__NULL__"

    return (
        type(v).__name__
        + "::"
        + repr(v)
    )

def avaliar_associacao(
    df,
    coluna_codigo,
    coluna_texto,
    cardinalidade_codigo,
    score_nome,
):
    pares = (
        df.select(
            coluna_codigo,
            coluna_texto,
        )
        .where(
            F.col(
                coluna_codigo
            ).isNotNull()
        )
        .groupBy(
            coluna_codigo,
            coluna_texto,
        )
        .agg(
            F.count("*").alias(
                "quantidade"
            )
        )
        .persist(
            StorageLevel.MEMORY_AND_DISK
        )
    )

    por_codigo = (
        pares.groupBy(
            coluna_codigo
        )
        .agg(
            F.sum(
                F.when(
                    F.col(
                        coluna_texto
                    ).isNotNull(),
                    1,
                ).otherwise(0)
            ).alias(
                "qt_textos_nao_nulos"
            ),
            F.sum(
                F.when(
                    F.col(
                        coluna_texto
                    ).isNull(),
                    F.col(
                        "quantidade"
                    ),
                ).otherwise(0)
            ).alias(
                "linhas_texto_nulo"
            ),
        )
    )

    stats = (
        por_codigo.agg(
            F.sum(
                F.when(
                    F.col(
                        "qt_textos_nao_nulos"
                    ) > 1,
                    1,
                ).otherwise(0)
            ).alias("MULTI"),
            F.sum(
                F.when(
                    F.col(
                        "qt_textos_nao_nulos"
                    ) == 1,
                    1,
                ).otherwise(0)
            ).alias("UM"),
            F.sum(
                F.when(
                    F.col(
                        "qt_textos_nao_nulos"
                    ) == 0,
                    1,
                ).otherwise(0)
            ).alias("SEM"),
            F.sum(
                F.when(
                    F.col(
                        "linhas_texto_nulo"
                    ) > 0,
                    1,
                ).otherwise(0)
            ).alias("NULO"),
        )
        .first()
    )

    multi = int(
        stats["MULTI"]
        or 0
    )
    um = int(
        stats["UM"]
        or 0
    )
    sem = int(
        stats["SEM"]
        or 0
    )
    nulo = int(
        stats["NULO"]
        or 0
    )

    if (
        um == 0
        and multi == 0
    ):
        situacao = (
            "SEM DESCRIÇÃO ASSOCIADA"
        )

    elif multi > 0:
        situacao = (
            "NÃO UNÍVOCA"
        )

    elif (
        sem > 0
        or nulo > 0
        or um
        < int(
            cardinalidade_codigo
        )
    ):
        situacao = "PARCIAL"

    else:
        situacao = "UNÍVOCA"

    # O domínio tem <=100 valores; portanto esta coleta é pequena.
    linhas = (
        pares.orderBy(
            F.col(
                coluna_codigo
            ).asc_nulls_last(),
            F.desc(
                "quantidade"
            ),
        )
        .limit(500)
        .collect()
    )

    mapa = defaultdict(list)

    for row in linhas:
        codigo = row[
            coluna_codigo
        ]
        texto = row[
            coluna_texto
        ]

        mapa[
            chave_valor(codigo)
        ].append(
            {
                "codigo": (
                    valor_python(
                        codigo
                    )
                ),
                "texto": (
                    sanitizar_texto(
                        texto
                    )
                    if texto is not None
                    else None
                ),
                "quantidade": int(
                    row[
                        "quantidade"
                    ]
                    or 0
                ),
            }
        )

    consolidado = {}

    for k, itens in mapa.items():
        textos = list(
            dict.fromkeys(
                x["texto"]
                for x in itens
                if x["texto"]
                is not None
            )
        )

        consolidado[k] = {
            "codigo": itens[0][
                "codigo"
            ],
            "textos": textos,
            "texto_inequivoco": (
                textos[0]
                if len(textos) == 1
                else None
            ),
            "tem_nulo": any(
                x["texto"]
                is None
                for x in itens
            ),
        }

    combinacoes = []

    if situacao == "NÃO UNÍVOCA":
        for itens in mapa.values():
            for x in itens:
                combinacoes.append(
                    x
                )

    pares.unpersist()

    return {
        "coluna_codigo": coluna_codigo,
        "coluna_texto": coluna_texto,
        "score_nome": score_nome,
        "situacao": situacao,
        "codigos_um_texto": um,
        "codigos_multiplos_textos": multi,
        "codigos_sem_texto": sem,
        "codigos_com_nulo": nulo,
        "mapeamento": consolidado,
        "combinacoes_nao_univocas": (
            combinacoes[:100]
        ),
    }

def descobrir_associacoes(
    df,
    perfil,
):
    idx = {
        p["nome"]: p
        for p in perfil
    }

    saida = []

    for cand in candidatos_associacao(
        perfil
    ):
        codigo = cand[
            "codigo"
        ]

        card = idx[codigo][
            "cardinalidade"
        ]

        if card == ">100":
            continue

        saida.append(
            avaliar_associacao(
                df,
                codigo,
                cand["texto"],
                int(card),
                cand[
                    "score_nome"
                ],
            )
        )

    return saida

def ranking_associacao(a):
    peso = {
        "UNÍVOCA": 4,
        "PARCIAL": 3,
        "NÃO UNÍVOCA": 2,
        "SEM DESCRIÇÃO ASSOCIADA": 1,
    }.get(
        a["situacao"],
        0,
    )

    return (
        peso,
        a.get(
            "codigos_um_texto",
            0,
        ),
        a.get(
            "score_nome",
            0,
        ),
        -a.get(
            "codigos_multiplos_textos",
            0,
        ),
    )

def construir_dominios(
    tabela,
    perfil,
    associacoes,
):
    dominios = []

    for p in perfil:
        if (
            p["classificacao"]
            != "DOMÍNIO / CÓDIGO"
        ):
            continue

        coluna = p["nome"]

        candidatas = [
            a
            for a in associacoes
            if (
                a[
                    "coluna_codigo"
                ]
                == coluna
            )
        ]

        candidatas.sort(
            key=ranking_associacao,
            reverse=True,
        )

        melhor = (
            candidatas[0]
            if candidatas
            else None
        )

        valores = []

        for freq in p[
            "valores"
        ]:
            codigo = freq.get(
                coluna
            )

            quantidade = int(
                freq.get(
                    "quantidade",
                    0,
                )
            )

            percentual = float(
                freq.get(
                    "percentual",
                    0.0,
                )
            )

            if codigo is None:
                valores.append(
                    {
                        "codigo": None,
                        "texto_observado": None,
                        "quantidade": quantidade,
                        "percentual": percentual,
                        "significado": "ausência de valor",
                    }
                )
                continue

            texto_obs = None
            significado = (
                "[PREENCHER]"
            )

            if melhor:
                item = melhor[
                    "mapeamento"
                ].get(
                    chave_valor(
                        codigo
                    )
                )

                if item:
                    texto_obs = item.get(
                        "texto_inequivoco"
                    )

                    if (
                        texto_obs
                        is not None
                    ):
                        significado = (
                            texto_obs
                        )

            valores.append(
                {
                    "codigo": (
                        valor_python(
                            codigo
                        )
                    ),
                    "texto_observado": texto_obs,
                    "quantidade": quantidade,
                    "percentual": percentual,
                    "significado": significado,
                }
            )

            if (
                significado
                == "[PREENCHER]"
            ):
                RESULTADO_ESTUDO[
                    "dicionario_pendente"
                ].append(
                    {
                        "tabela": tabela,
                        "coluna": coluna,
                        "codigo": (
                            valor_python(
                                codigo
                            )
                        ),
                        "frequencia": quantidade,
                        "texto_associado": (
                            texto_obs
                            if texto_obs
                            is not None
                            else "—"
                        ),
                        "significado": "[PREENCHER]",
                    }
                )

        dominios.append(
            {
                "coluna": coluna,
                "melhor_associacao": {
                    "coluna_texto": (
                        melhor[
                            "coluna_texto"
                        ]
                        if melhor
                        else None
                    ),
                    "situacao": (
                        melhor[
                            "situacao"
                        ]
                        if melhor
                        else (
                            "SEM DESCRIÇÃO ASSOCIADA"
                        )
                    ),
                },
                "valores": valores,
            }
        )

    return dominios

def consolidar_inferidos(
    tabela,
    associacoes,
):
    for a in associacoes:
        if a["situacao"] not in (
            "UNÍVOCA",
            "PARCIAL",
        ):
            continue

        for item in a[
            "mapeamento"
        ].values():
            texto = item.get(
                "texto_inequivoco"
            )

            if texto is None:
                continue

            RESULTADO_ESTUDO[
                "dicionario_inferido"
            ].append(
                {
                    "tabela": tabela,
                    "coluna_codigo": (
                        a[
                            "coluna_codigo"
                        ]
                    ),
                    "codigo": item[
                        "codigo"
                    ],
                    "coluna_texto": (
                        a[
                            "coluna_texto"
                        ]
                    ),
                    "texto_observado": texto,
                    "situacao_associacao": (
                        a[
                            "situacao"
                        ]
                    ),
                }
            )

print("[OK] Associação intratabela carregada.")

In [ ]:
%%spark

# ============================================================
# 7. Execução das seis tabelas
# ============================================================

def resumo_classificacoes(
    perfil,
):
    cont = defaultdict(int)

    for p in perfil:
        cont[
            p["classificacao"]
        ] += 1

    return dict(cont)

def periodo_observado(
    perfil,
):
    return [
        {
            "coluna": p["nome"],
            "menor": p["minimo"],
            "maior": p["maximo"],
            "nulos": p["nulos"],
        }
        for p in perfil
        if (
            p["classificacao"]
            == "DATA / TIMESTAMP"
            and p["minimo"]
            is not None
        )
    ]

def perfilar_tabela(
    tabela,
):
    probe, df, leitura = (
        carregar_tabela(
            tabela
        )
    )

    df = df.persist(
        StorageLevel.MEMORY_AND_DISK
    )

    total = int(
        df.count()
    )

    perfil = construir_perfil(
        df,
        total,
    )

    associacoes = (
        descobrir_associacoes(
            df,
            perfil,
        )
    )

    dominios = construir_dominios(
        tabela,
        perfil,
        associacoes,
    )

    consolidar_inferidos(
        tabela,
        associacoes,
    )

    resumo = (
        resumo_classificacoes(
            perfil
        )
    )

    print(
        f"[TABELA] {tabela}"
    )
    print(
        "Universo: "
        + leitura[
            "universo_analisado"
        ]
    )
    print(
        f"Linhas analisadas: {total}"
    )
    print(
        f"Colunas: {len(probe.columns)}"
    )
    print(
        "Domínios: "
        f"{resumo.get('DOMÍNIO / CÓDIGO', 0)}"
    )
    print("Concluído.")

    resultado = {
        "tabela": tabela,
        "grupo": CONFIG[
            tabela
        ]["grupo"],
        "status": "ANALISADA",
        "documentacao_projeto": CONFIG[
            tabela
        ][
            "documentacao_projeto"
        ],
        "universo_planejado": CONFIG[
            tabela
        ][
            "universo_planejado"
        ],
        "universo_analisado": leitura[
            "universo_analisado"
        ],
        "volume_total_tabela": leitura.get(
            "volume_total_tabela"
        ),
        "amostra_limitada": leitura.get(
            "amostra_limitada",
            False,
        ),
        "motivo_amostra": leitura.get(
            "motivo_amostra"
        ),
        "linhas_analisadas": total,
        "quantidade_colunas": len(
            probe.columns
        ),
        "resumo_classificacoes": resumo,
        "periodo": periodo_observado(
            perfil
        ),
        "perfil_colunas": perfil,
        "dominios": dominios,
        "associacoes_codigo_texto": [
            {
                k: v
                for k, v in a.items()
                if k != "mapeamento"
            }
            for a in associacoes
        ],
    }

    df.unpersist()

    return resultado

for tabela in TABELAS:
    try:
        RESULTADO_ESTUDO[
            "tabelas"
        ][tabela] = (
            perfilar_tabela(
                tabela
            )
        )

    except Exception as exc:
        RESULTADO_ESTUDO[
            "execucao_ok"
        ] = False

        codigo = getattr(
            exc,
            "codigo",
            None,
        )

        motivo = (
            type(exc).__name__
            + (
                f" [{codigo}]"
                if codigo
                else ""
            )
        )

        RESULTADO_ESTUDO[
            "tabelas"
        ][tabela] = {
            "tabela": tabela,
            "grupo": CONFIG[
                tabela
            ]["grupo"],
            "status": (
                "NÃO ANALISADA"
            ),
            "documentacao_projeto": CONFIG[
                tabela
            ][
                "documentacao_projeto"
            ],
            "universo_planejado": CONFIG[
                tabela
            ][
                "universo_planejado"
            ],
            "universo_analisado": (
                "NÃO DETERMINADO"
            ),
            "motivo_tecnico": motivo,
            "perfil_colunas": [],
            "dominios": [],
            "associacoes_codigo_texto": [],
        }

        RESULTADO_ESTUDO[
            "observacoes_tecnicas"
        ].append(
            {
                "tabela": tabela,
                "observacao": (
                    "Falha parcial: "
                    + motivo
                ),
            }
        )

        print(
            f"[TABELA] {tabela}"
        )
        print(
            "STATUS: NÃO ANALISADA"
        )
        print(
            f"MOTIVO TÉCNICO: {motivo}"
        )

print("[FIM] Perfilamento remoto concluído.")

In [ ]:
# ============================================================
# 8. Markdown final
# ============================================================
import re

def fmt_int(v):
    if v is None:
        return "NÃO DETERMINADO"

    return (
        f"{int(v):,}"
        .replace(",", ".")
    )

def fmt_pct(v):
    if v is None:
        return ""

    return (
        f"{float(v):.6f}"
        .rstrip("0")
        .rstrip(".")
        .replace(".", ",")
        + "%"
    )

def fmt_num(v):
    if v is None:
        return ""

    if isinstance(v, float):
        return (
            f"{v:.6f}"
            .rstrip("0")
            .rstrip(".")
            .replace(".", ",")
        )

    return str(v)

def render_valor(v):
    if v is None:
        return "`NULL`"

    if isinstance(v, str):
        return (
            "`"
            + repr(v).replace(
                "`",
                "\\`",
            )
            + "`"
        )

    return (
        "`"
        + str(v)
        + "`"
    )

def safe_text(v):
    if v is None:
        return ""

    texto = str(v)

    texto = re.sub(
        r"\b\d{3}\.?\d{3}\.?\d{3}-?\d{2}\b",
        "<DOCUMENTO_OCULTO>",
        texto,
    )
    texto = re.sub(
        r"\b\d{2}\.?\d{3}\.?\d{3}/?\d{4}-?\d{2}\b",
        "<DOCUMENTO_OCULTO>",
        texto,
    )

    return (
        texto
        .replace("|", "\\|")
        .replace("\n", " ")
    )

def tabela_md(
    rows,
    colunas,
):
    if not rows:
        return "_Nenhum registro._"

    linhas = [
        "| "
        + " | ".join(
            colunas
        )
        + " |",
        "| "
        + " | ".join(
            "---"
            for _ in colunas
        )
        + " |",
    ]

    for row in rows:
        linhas.append(
            "| "
            + " | ".join(
                safe_text(
                    row.get(
                        c,
                        "",
                    )
                )
                for c in colunas
            )
            + " |"
        )

    return "\n".join(
        linhas
    )

def render_secao_tabela(
    linhas,
    tabela,
    t,
):
    linhas += [
        f"## {tabela}",
        "",
        "### Resumo da tabela",
        "",
        (
            "**Documentação no Projeto:** "
            + safe_text(
                t.get(
                    "documentacao_projeto"
                )
            )
        ),
        "",
        (
            "**Universo planejado:** "
            + safe_text(
                t.get(
                    "universo_planejado"
                )
            )
        ),
        "",
        (
            "**Universo efetivamente analisado:** "
            + safe_text(
                t.get(
                    "universo_analisado"
                )
            )
        ),
        "",
    ]

    if (
        t.get("status")
        != "ANALISADA"
    ):
        linhas += [
            "**STATUS: NÃO ANALISADA**",
            "",
            (
                "**MOTIVO TÉCNICO:** "
                + safe_text(
                    t.get(
                        "motivo_tecnico"
                    )
                )
            ),
            "",
        ]
        return

    if (
        t.get(
            "volume_total_tabela"
        )
        is not None
    ):
        linhas += [
            (
                "**Volume total medido antes do perfil:** "
                f"{fmt_int(t['volume_total_tabela'])}"
            ),
            "",
        ]

    if t.get(
        "amostra_limitada"
    ):
        linhas += [
            (
                "**ATENÇÃO — amostra técnica:** "
                + safe_text(
                    t.get(
                        "motivo_amostra"
                    )
                )
            ),
            "",
            (
                "Os valores observados abaixo descrevem o recorte "
                "e não devem ser apresentados como universo histórico completo."
            ),
            "",
        ]

    rc = t.get(
        "resumo_classificacoes",
        {},
    )

    linhas += [
        (
            "**Linhas analisadas:** "
            f"{fmt_int(t.get('linhas_analisadas'))}"
        ),
        "",
        (
            "**Colunas:** "
            f"{fmt_int(t.get('quantidade_colunas'))}"
        ),
        "",
        (
            "**Domínios/códigos:** "
            f"{rc.get('DOMÍNIO / CÓDIGO', 0)}"
        ),
        "",
        (
            "**Identificadores:** "
            f"{rc.get('IDENTIFICADOR', 0)}"
        ),
        "",
        (
            "**Textos/descrições:** "
            f"{rc.get('TEXTO / DESCRIÇÃO', 0)}"
        ),
        "",
        (
            "**Datas/timestamps:** "
            f"{rc.get('DATA / TIMESTAMP', 0)}"
        ),
        "",
        (
            "**Numéricos contínuos:** "
            f"{rc.get('NUMÉRICO CONTÍNUO', 0)}"
        ),
        "",
        "### Inventário de colunas",
        "",
    ]

    inventario = []

    for p in t.get(
        "perfil_colunas",
        [],
    ):
        inventario.append(
            {
                "Coluna": p[
                    "nome"
                ],
                "Tipo": p[
                    "tipo"
                ],
                "Preenchidos": fmt_int(
                    p[
                        "preenchidos"
                    ]
                ),
                "Nulos": fmt_int(
                    p["nulos"]
                ),
                "% preenchido": fmt_pct(
                    p[
                        "pct_preenchido"
                    ]
                ),
                "Cardinalidade": p[
                    "cardinalidade"
                ],
                "Classificação": p[
                    "classificacao"
                ],
            }
        )

    linhas += [
        tabela_md(
            inventario,
            [
                "Coluna",
                "Tipo",
                "Preenchidos",
                "Nulos",
                "% preenchido",
                "Cardinalidade",
                "Classificação",
            ],
        ),
        "",
        "### Domínios e códigos",
        "",
    ]

    if not t.get("dominios"):
        linhas += [
            "_Nenhuma coluna classificada automaticamente como domínio/código._",
            "",
        ]

    for d in t.get(
        "dominios",
        [],
    ):
        linhas += [
            f"#### `{d['coluna']}`",
            "",
            (
                "**Coluna textual associada:** "
                + safe_text(
                    d[
                        "melhor_associacao"
                    ].get(
                        "coluna_texto"
                    )
                    or "não encontrada"
                )
            ),
            "",
            (
                "**Situação da associação:** "
                + safe_text(
                    d[
                        "melhor_associacao"
                    ].get(
                        "situacao"
                    )
                )
            ),
            "",
        ]

        rows_dom = []

        for item in d[
            "valores"
        ]:
            rows_dom.append(
                {
                    "Código": render_valor(
                        item[
                            "codigo"
                        ]
                    ),
                    "Texto observado": (
                        render_valor(
                            item[
                                "texto_observado"
                            ]
                        )
                        if item.get(
                            "texto_observado"
                        )
                        is not None
                        else "—"
                    ),
                    "Quantidade": fmt_int(
                        item[
                            "quantidade"
                        ]
                    ),
                    "%": fmt_pct(
                        item[
                            "percentual"
                        ]
                    ),
                    "Significado": safe_text(
                        item[
                            "significado"
                        ]
                    ),
                }
            )

        linhas += [
            tabela_md(
                rows_dom,
                [
                    "Código",
                    "Texto observado",
                    "Quantidade",
                    "%",
                    "Significado",
                ],
            ),
            "",
        ]

    linhas += [
        "### Associações código ↔ texto",
        "",
    ]

    assoc_rows = []

    for a in t.get(
        "associacoes_codigo_texto",
        [],
    ):
        assoc_rows.append(
            {
                "Código": a[
                    "coluna_codigo"
                ],
                "Texto": a[
                    "coluna_texto"
                ],
                "Situação": a[
                    "situacao"
                ],
                "1 texto": a[
                    "codigos_um_texto"
                ],
                ">1 texto": a[
                    "codigos_multiplos_textos"
                ],
                "Sem texto": a[
                    "codigos_sem_texto"
                ],
            }
        )

    linhas += [
        tabela_md(
            assoc_rows,
            [
                "Código",
                "Texto",
                "Situação",
                "1 texto",
                ">1 texto",
                "Sem texto",
            ],
        ),
        "",
    ]

    for a in t.get(
        "associacoes_codigo_texto",
        [],
    ):
        if (
            a.get(
                "situacao"
            )
            != "NÃO UNÍVOCA"
        ):
            continue

        linhas += [
            (
                "#### Associação não unívoca: "
                f"`{a['coluna_codigo']}` ↔ "
                f"`{a['coluna_texto']}`"
            ),
            "",
        ]

        combos = []

        for x in a.get(
            "combinacoes_nao_univocas",
            [],
        ):
            combos.append(
                {
                    "Código": render_valor(
                        x[
                            "codigo"
                        ]
                    ),
                    "Texto": (
                        render_valor(
                            x[
                                "texto"
                            ]
                        )
                        if x.get(
                            "texto"
                        )
                        is not None
                        else "`NULL`"
                    ),
                    "Quantidade": fmt_int(
                        x[
                            "quantidade"
                        ]
                    ),
                }
            )

        linhas += [
            tabela_md(
                combos,
                [
                    "Código",
                    "Texto",
                    "Quantidade",
                ],
            ),
            "",
        ]

    linhas += [
        "### Datas",
        "",
    ]

    datas = []

    for p in t.get(
        "perfil_colunas",
        [],
    ):
        if (
            p["classificacao"]
            != "DATA / TIMESTAMP"
        ):
            continue

        datas.append(
            {
                "Coluna": p[
                    "nome"
                ],
                "Menor": p[
                    "minimo"
                ]
                or "",
                "Maior": p[
                    "maximo"
                ]
                or "",
                "Nulos": fmt_int(
                    p["nulos"]
                ),
            }
        )

    linhas += [
        tabela_md(
            datas,
            [
                "Coluna",
                "Menor",
                "Maior",
                "Nulos",
            ],
        ),
        "",
        "### Identificadores / alta cardinalidade",
        "",
    ]

    altas = []

    for p in t.get(
        "perfil_colunas",
        [],
    ):
        if not (
            p[
                "classificacao"
            ]
            == "IDENTIFICADOR"
            or p[
                "cardinalidade"
            ]
            == ">100"
        ):
            continue

        obs = []

        if (
            p[
                "classificacao"
            ]
            == "IDENTIFICADOR"
        ):
            obs.append(
                "valores individuais não expostos"
            )

        if (
            p[
                "cardinalidade"
            ]
            == ">100"
        ):
            obs.append(
                "cardinalidade exata não calculada"
            )

        altas.append(
            {
                "Coluna": p[
                    "nome"
                ],
                "Cardinalidade": p[
                    "cardinalidade"
                ],
                "Preenchimento": fmt_pct(
                    p[
                        "pct_preenchido"
                    ]
                ),
                "Observação": "; ".join(
                    obs
                ),
            }
        )

    linhas += [
        tabela_md(
            altas,
            [
                "Coluna",
                "Cardinalidade",
                "Preenchimento",
                "Observação",
            ],
        ),
        "",
        "### Estatísticas técnicas complementares",
        "",
    ]

    extras = []

    for p in t.get(
        "perfil_colunas",
        [],
    ):
        if (
            p[
                "classificacao"
            ]
            == "TEXTO / DESCRIÇÃO"
        ):
            extras.append(
                {
                    "Coluna": p[
                        "nome"
                    ],
                    "Classe": p[
                        "classificacao"
                    ],
                    "Mínimo": "",
                    "Máximo": "",
                    "Média": "",
                    "Compr. mín.": p[
                        "comprimento_min"
                    ],
                    "Compr. médio": fmt_num(
                        p[
                            "comprimento_medio"
                        ]
                    ),
                    "Compr. máx.": p[
                        "comprimento_max"
                    ],
                    "Strings vazias": fmt_int(
                        p[
                            "strings_vazias"
                        ]
                    ),
                    "Somente espaços": fmt_int(
                        p[
                            "somente_espacos"
                        ]
                    ),
                }
            )

        elif (
            p[
                "classificacao"
            ]
            == "NUMÉRICO CONTÍNUO"
        ):
            extras.append(
                {
                    "Coluna": p[
                        "nome"
                    ],
                    "Classe": p[
                        "classificacao"
                    ],
                    "Mínimo": fmt_num(
                        p["minimo"]
                    ),
                    "Máximo": fmt_num(
                        p["maximo"]
                    ),
                    "Média": fmt_num(
                        p["media"]
                    ),
                    "Compr. mín.": "",
                    "Compr. médio": "",
                    "Compr. máx.": "",
                    "Strings vazias": "",
                    "Somente espaços": "",
                }
            )

    linhas += [
        tabela_md(
            extras,
            [
                "Coluna",
                "Classe",
                "Mínimo",
                "Máximo",
                "Média",
                "Compr. mín.",
                "Compr. médio",
                "Compr. máx.",
                "Strings vazias",
                "Somente espaços",
            ],
        ),
        "",
        "### Pontos para preencher manualmente",
        "",
    ]

    pendentes = [
        x
        for x in RESULTADO_ESTUDO.get(
            "dicionario_pendente",
            [],
        )
        if x["tabela"] == tabela
    ]

    if not pendentes:
        linhas += [
            "_Nenhum código pendente nesta tabela._",
            "",
        ]

    else:
        por_coluna = {}

        for x in pendentes:
            por_coluna.setdefault(
                x["coluna"],
                [],
            ).append(x)

        for coluna, itens in por_coluna.items():
            linhas.append(
                f"- `{coluna}`"
            )

            for x in itens:
                linhas.append(
                    "  - "
                    + render_valor(
                        x[
                            "codigo"
                        ]
                    )
                    + " → `[PREENCHER]`"
                )

        linhas.append("")

try:
    RESULTADO_ESTUDO = (
        spark.get_from_spark(
            "RESULTADO_ESTUDO"
        )
    )
except Exception as exc:
    OUTPUT_MD.write_text(
        "# Estudo das tabelas\n\n"
        "## Observações técnicas\n\n"
        "- **EXECUÇÃO INCOMPLETA** — "
        f"resultado Spark indisponível ({type(exc).__name__}).\n",
        encoding="utf-8",
    )
    raise

linhas = [
    "# Estudo das tabelas",
    "",
    (
        "Dicionário empírico construído exclusivamente "
        "a partir dos valores observados."
    ),
    "",
    "# Fontes principais",
    "",
]

for tabela in [
    "DB2GFP.TRAN_RLZD_INST_PCT",
    "DB2GFP.INF_OPB_CT_CLI",
    "DB2GFP.CMPT_TRAN_RLZD_CC",
    "DB2GFP.CTGR_TRAN_OPB",
    "DB2GFP.GR_CTGR_TRAN",
]:
    render_secao_tabela(
        linhas,
        tabela,
        RESULTADO_ESTUDO[
            "tabelas"
        ].get(
            tabela,
            {},
        ),
    )

linhas += [
    "# Estudo adicional — DB2OPB.AUTZ_ATV_CPTO_CLI",
    "",
    (
        "Esta tabela foi incluída separadamente. "
        "Não foi encontrada documentação específica dela nos arquivos do Projeto; "
        "portanto, a seção abaixo é exclusivamente empírica."
    ),
    "",
]

render_secao_tabela(
    linhas,
    "DB2OPB.AUTZ_ATV_CPTO_CLI",
    RESULTADO_ESTUDO[
        "tabelas"
    ].get(
        "DB2OPB.AUTZ_ATV_CPTO_CLI",
        {},
    ),
)

linhas += [
    "# Dicionário pendente de significado",
    "",
]

pend_rows = []

for x in RESULTADO_ESTUDO.get(
    "dicionario_pendente",
    [],
):
    pend_rows.append(
        {
            "Tabela": x[
                "tabela"
            ],
            "Coluna": x[
                "coluna"
            ],
            "Código": render_valor(
                x[
                    "codigo"
                ]
            ),
            "Frequência": fmt_int(
                x[
                    "frequencia"
                ]
            ),
            "Texto associado": safe_text(
                x[
                    "texto_associado"
                ]
            ),
            "Significado": "[PREENCHER]",
        }
    )

linhas += [
    tabela_md(
        pend_rows,
        [
            "Tabela",
            "Coluna",
            "Código",
            "Frequência",
            "Texto associado",
            "Significado",
        ],
    ),
    "",
    "# Dicionário inferido diretamente da própria tabela",
    "",
]

inf_rows = []

for x in RESULTADO_ESTUDO.get(
    "dicionario_inferido",
    [],
):
    inf_rows.append(
        {
            "Tabela": x[
                "tabela"
            ],
            "Coluna código": x[
                "coluna_codigo"
            ],
            "Código": render_valor(
                x[
                    "codigo"
                ]
            ),
            "Coluna texto": x[
                "coluna_texto"
            ],
            "Texto observado": safe_text(
                x[
                    "texto_observado"
                ]
            ),
            "Situação": x[
                "situacao_associacao"
            ],
        }
    )

linhas += [
    tabela_md(
        inf_rows,
        [
            "Tabela",
            "Coluna código",
            "Código",
            "Coluna texto",
            "Texto observado",
            "Situação",
        ],
    ),
    "",
    "# Observações técnicas",
    "",
    (
        "- As seis tabelas foram tratadas isoladamente; "
        "não existem joins entre fontes."
    ),
    (
        "- `TRAN_RLZD_INST_PCT` usa recorte técnico dos 3 últimos meses completos por `DT_TRAN`."
    ),
    (
        "- As outras fontes usam política adaptativa: tabela completa "
        "quando o volume é tecnicamente aceitável; caso contrário, amostra técnica explícita."
    ),
    (
        "- `DB2OPB.AUTZ_ATV_CPTO_CLI` é seção adicional e não foi incorporada "
        "às cinco fontes principais."
    ),
    (
        "- A cardinalidade aproximada de todas as colunas é calculada em uma única agregação."
    ),
    (
        "- Cardinalidade exata é calculada apenas para candidatas próximas ao limiar de domínio."
    ),
    (
        "- Colunas já classificadas como alta cardinalidade não recebem groupBy de top valores."
    ),
    (
        "- Associações código ↔ texto são exclusivamente intratabela e são testadas nos próprios dados."
    ),
    (
        "- Nenhum significado externo foi acrescentado."
    ),
]

for obs in RESULTADO_ESTUDO.get(
    "observacoes_tecnicas",
    [],
):
    linhas.append(
        "- "
        + safe_text(
            obs.get(
                "tabela"
            )
        )
        + ": "
        + safe_text(
            obs.get(
                "observacao"
            )
        )
    )

if not RESULTADO_ESTUDO.get(
    "execucao_ok"
):
    linhas += [
        "",
        (
            "**EXECUÇÃO INCOMPLETA:** uma ou mais tabelas não puderam "
            "ser analisadas. As demais seções foram preservadas."
        ),
    ]

OUTPUT_MD.write_text(
    "\n".join(linhas).strip()
    + "\n",
    encoding="utf-8",
)

print(
    f"[OK] Gerado: {OUTPUT_MD}"
)

## Produto final

Execute o notebook e revise:

`estudo_tabelas_resultado.md`

A tabela `DB2OPB.AUTZ_ATV_CPTO_CLI` aparecerá em uma seção adicional isolada no mesmo Markdown.